# Scube Mouse Hypothalamus (2/2): Alignment of consecutive ST slices

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import os

import warnings
warnings.filterwarnings("ignore")

In [2]:
import SPACEL
from SPACEL import Scube

In [3]:
seed = 1
from SPACEL.setting import set_environ_seed
set_environ_seed(seed)

Setting environment seed: 1


## Spatial transcriptomic data import

The input data type of Scube is a list of anndata objects for each slice. The anndata objects of each slice must contain a column of spatial domain for alignment in ``.obs``.

In [ ]:
adata = sc.read(f'./{seed}/simData_151673_all_Splane.h5ad')

adata[0].obs

,in_tissue,array_row,array_col,original_clusters,SpotIndex,0,1,2,3,4,5,6,slice_id,spatial_domain
AAACAAGTATCTCCCA-1,1,50,102,3,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,151673,1


In [11]:
slices = ['151673', '100', '80', '60', '40', '20']

adata_list=[]

for i in range(6):
    adata_list.append(adata[adata.obs['slice_id'] == slices[i]])
    adata_list[i].obs.spatial_domain = adata_list[i].obs.spatial_domain.astype('category')

In [12]:
adata_list[1]

AnnData object with n_obs × n_vars = 3596 × 33538
    obs: 'in_tissue', 'array_row', 'array_col', 'original_clusters', 'SpotIndex', '0', '1', '2', '3', '4', '5', '6', 'slice_id', 'spatial_domain'
    obsm: 'spatial'

Plot all raw slices stacked in a single figure. Cells colored by spatial domain.First, we define the color of each spatial domain.

## Pairwise alignment

Here, we perform pairwise alignment of slices. The ``cluster_key`` is a column of ``.obs`` which contain spatial domain information used for alignment. The ``n_neighbors`` affect how much neighbors are considered when calculate similarity of two slices. The ``knn_exclude_cutoff`` affect max distance of neighbors. By default, ``n_neighbors = 15`` and ``knn_exclude_cutoff = n_neighbors`` means a spot/cell in source slices consider 15 neighbors in target slices, and neighbor will be exclude when the distance of neighbors larger than the median of neareast 30 neighbors distance in all spot/cell in target slice. ``p`` affect the exponent of overlap penalty, and a larger p means stronger partial alignment capability.

In [ ]:
Scube.align(adata_list,
      cluster_key='spatial_domain', 
      n_neighbors = 15, 
      n_threads=10,
      p=2,
      seed=seed,
     )

Start alignment...
Alignment slice 1 to 0
Alignment slice 2 to 1
Alignment slice 3 to 2
Alignment slice 4 to 3
Alignment slice 5 to 4
Runtime: 55.49591517448425 s


After alignment, the new spots/cells location coordinates saved in ``.obsm['spatial_aligned']`` of anndata object of each slices. When ``write_loc_path`` provied by users, the location coordinates of all spots/cells of all slices will be saved in ``write_loc_path``.

## LAA & ARI

In [14]:
from sklearn.metrics import pairwise
from sklearn.metrics import adjusted_rand_score

# LAA
#calculate laywise bar, result needs to be devided by total number to calculate the accuracy rate
def cal_layer_based_alignment_result(alignment, s1, s2):
    #fei added
    labels = []
    labels.extend(s1.obs['original_clusters'])
    labels.extend(s2.obs['original_clusters'])

    res = []
    l_dict = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6}
    cnt0 = 0
    cnt1 = 0
    cnt2 = 0
    cnt3 = 0
    cnt4 = 0
    cnt5 = 0
    cnt6 = 0
    for i, elem in enumerate(alignment):
        if np.max(elem) - np.min(elem) != 0: # paste2 might have spots that are not aligned to any other spots. paste might have negative alignment
            if labels[i] == '-1' or labels[elem.argmax() + alignment.shape[0]] == '-1':
                continue
            if l_dict[labels[i]] == l_dict[labels[elem.argmax() + alignment.shape[0]]]:
                cnt0 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 1:
                cnt1 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 2:
                cnt2 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 3:
                cnt3 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 4:
                cnt4 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 5:
                cnt5 += 1
            if abs(l_dict[labels[i]] - l_dict[labels[elem.argmax() + alignment.shape[0]]]) == 6:
                cnt6 += 1
    res.extend([cnt0, cnt1, cnt2, cnt3, cnt4, cnt5, cnt6])
    return res


def compute_alignment_ari(sliceA, sliceB, pi):
    mapped_clusters = []
    for j in range(pi.shape[1]):
        mapping = pi[:, j]
        #original: if np.sum(mapping) > 0:
        if np.max(mapping) - np.min(mapping) != 0:
            i = np.argmax(mapping)
            mapped_clusters.append(sliceA.obs['original_clusters'][i])
        else:
            mapped_clusters.append("NULL")
    assert len(sliceB.obs['original_clusters']) == len(mapped_clusters)
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []
    for i in range(len(sliceB.obs['original_clusters'])):
        if mapped_clusters[i] != "NULL":
            true_clusters_mapped_region.append(sliceB.obs['original_clusters'][i])
            mapped_clusters_mapped_region.append(mapped_clusters[i])

    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
    return ari

In [ ]:
ari_values = []
laa_values = []

for i in range(1, len(slices)):
    target_ind = i-1
    source_ind = i
    f1 = adata_list[target_ind].obsm['spatial_aligned'].to_numpy()
    f2 = adata_list[source_ind].obsm['spatial_aligned'].to_numpy()

    alignment = pairwise.euclidean_distances(f1, f2)
    alignment = np.exp(-alignment)
    
    name1 = slices[target_ind]
    name2 = slices[source_ind]
    # Save as .pkl
    alignment_df = pd.DataFrame(alignment)
    alignment_df.to_pickle(f'./{seed}/{name1}_{name2}_AlignmentPi.pkl')
    
    res = cal_layer_based_alignment_result(alignment, adata_list[target_ind], adata_list[source_ind])
    divisor = alignment.shape[0]
    laa_result = [num / divisor for num in res]
    
    ari = compute_alignment_ari(adata_list[target_ind], adata_list[source_ind], alignment)
    ari_values.append(ari)
    laa_values.append(laa_result)
    print(laa_result, ari)
    transposed = list(zip(*laa_values))

# Create DataFrame with desired format
header_row = [f"{slices[i-1]}_{slices[i]}" for i in range(1, len(slices))]
ari_df = pd.DataFrame([ari_values], columns=header_row)
laa_df = pd.DataFrame(transposed, columns=header_row)

[0.9905843256715591, 0.006092495153697037, 0.0, 0.0, 0.0, 0.0, 0.0] 0.9843079650260088
[0.7755839822024472, 0.029477196885428252, 0.0, 0.0, 0.0, 0.0, 0.004449388209121246] 0.9162455956855501
[0.7360055286800277, 0.033863165169315826, 0.0, 0.0, 0.0, 0.0, 0.0024187975120939877] 0.9052680727280563
[0.6321100917431193, 0.044036697247706424, 0.0, 0.0, 0.0, 0.0, 0.0] 0.8823897923322754
[0.46579129232895644, 0.028334485141672427, 0.0, 0.0, 0.0, 0.0, 0.0] 0.8883492839232082


In [ ]:
# Save as a CSV file
ari_df.to_csv(f'./{seed}/simu_ARI.csv')
laa_df.to_csv(f'./{seed}/simu_LAA.csv')

## Plot alignment results

Save all spots/cells aligned coordiantes and colors in a DataFrame.

In [ ]:
c_dict = {
    0: '#1f77b4',
    1: '#ff7f0e',
    2: '#2ca02c',
    3: '#d62728',
    4: '#9467bd',
    5: '#8c564b',
    6: '#e377c2'
    }

In [ ]:
coo = pd.DataFrame()

for i in range(len(adata_list)):
    loc = adata_list[i].obsm['spatial_aligned'].copy()
    loc['Z'] = i
    loc['slice'] = adata_list[i].obs['slice_id']
    loc['original_clusters'] = adata_list[i].obs['original_clusters']
    loc['celltype_colors'] = adata_list[i].obs['original_clusters'].replace(c_dict)
    coo = pd.concat([coo,loc],axis=0)
    
coo.to_csv(f'./{seed}/SPACEL_simu_aligned_coordinates_colors.csv')

In [22]:
coo

,X,Y,Z,slice,original_clusters,celltype_colors
AAACAAGTATCTCCCA-1,2817.000000,1701.000000,0,151673,3,#d62728
AAACAATCTACTAGCA-1,-1205.000000,-3960.000000,0,151673,1,#ff7f0e
AAACACCAATAACTGC-1,-2906.000000,2738.000000,0,151673,0,#1f77b4
AAACAGAGCGACTCCT-1,2297.000000,-2616.000000,0,151673,3,#d62728
AAACAGCTTTCAGAAG-1,-3581.000000,816.000000,0,151673,5,#8c564b
...,...,...,...,...,...,...
TTGCCGGTGATCCCTC-1,2637.270996,-2955.213867,5,20,3,#d62728
TTGCGGCGACTCATGC-1,480.563385,-4009.108643,5,20,2,#2ca02c
TTGCTGGCCGGGCTTC-1,1284.631592,-3277.041992,5,20,3,#d62728
TTGGATCGACTTCTGG-1,106.499092,-3136.762451,5,20,3,#d62728


In [ ]:
def plot_3d(
    loc,
    val=None,
    color=None,
    figsize=(12,12),
    return_fig=True,
    elev=None,
    azim=None,
    roll=None,
    xlim=None,
    ylim=None,
    zlim=None,
    frameon=True,
    save_path=None,
    save_dpi=150,
    show=True,
    *args,
    **kwargs
):
    """Plot all slices stacked in 3D

    Plot all slices stacked with a given number of subplots rows and columns. Spots/cells colored by spatial domain.

    Args:
        loc: An array of the coordinates of each spots/cells in all slices, which the first three columns are X-axis, Y-axis, Z-axis coordinates.
        val: The colors of each spots/cells given in loc use to plot.
        color: The colors of each spots/cells given in loc use to plot.
        figsize: Size of the figure.
        return_fig: Whether to return the figure.
        elev: The elevation angle in the vertical plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        azim: The azimuth angle in the horizontal plane in degrees. If ``None`` then the initial value as specified in the ``Axes3D`` constructor is used.
        xlim: A tuple given the left and right xlims in X-axis.
        ylim: A tuple given the left and right xlims in Y-axis.
        zlim: A tuple given the left and right xlims in Z-axis.
        frameon: Whether to hide the coordinate axes.
        save_path: A string representing the path directory where the figure saved.
        save_dpi: The resolution in dots per inch.
        show: Whether to show the figure.

    Returns:
        A ``matplotlib.figure.Figure`` object
    """
    if 'marker' not in kwargs.keys():
        kwargs['marker'] = 'o'
    if 's' not in kwargs.keys():
        kwargs['s'] = 5
    if 'cmap' not in kwargs.keys():
        kwargs['cmap'] = 'Spectral_r'
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(projection='3d')
    if (elev is not None) and (azim is not None) and (roll is not None):
        ax.view_init(elev, azim, roll)
    if color is None:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=val,*args,**kwargs)
    else:
        ax.scatter(loc[:,0], loc[:,1], loc[:,2],c=color,*args,**kwargs)
    
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    if not frameon:
        plt.axis('off')
        plt.grid(b=None)
    if save_path is not None:
        print(save_path)
        plt.savefig(save_path,dpi=save_dpi,bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()
    if return_fig:
        return fig

In [ ]:
df = pd.read_csv(f'./{seed}/SPACEL_simu_aligned_coordinates_colors.csv')
coords = df[['X', 'Y', 'Z']]
df

In [ ]:
fig = plot_3d(coords.values, val=None, color=df['celltype_colors'], elev=25, azim=75, roll=0, frameon=False, show=True, s=30, save_dpi=300, return_fig=True) # Scube is from SPACEL

fig.savefig(f'./{seed}/SPACEL_gtAndScube_simu_3dviz_angle25_75_s=30.png', bbox_inches="tight")

## Degree

In [2]:
def get_rotation_angle(R):
    # Extract the sine and cosine from the matrix
    cos_theta = R[0, 0]
    sin_theta = R[1, 0]

    # Compute the angle in radians
    theta = np.arctan2(sin_theta, cos_theta)

    # Convert the angle to degrees (optional)
    theta_deg = np.degrees(theta)

    return theta, theta_deg

In [337]:
seed = 2024

# https://github.com/raphael-group/paste/blob/main/src/paste/visualization.py
df = pd.read_csv(f'./{seed}/SPACEL_simu_aligned_coordinates_colors.csv')
coords = df.loc[df['Z'] == 0, ['X', 'Y']]
coords2 = df.loc[df['Z'] == 1, ['X', 'Y']]
alignment_df = pd.read_pickle(f'./{seed}/151673_100_AlignmentPi.pkl')
alignment = np.array(alignment_df)

X = np.array(coords)
print(X.shape)
Y = np.array(coords2)
print(Y.shape)
print(alignment.shape)
tX = alignment.sum(axis=1).dot(X)
tY = alignment.sum(axis=0).dot(Y)
X = X - tX
Y = Y - tY
H = Y.T.dot(alignment.T.dot(X))
U, S, Vt = np.linalg.svd(H)
R = Vt.T.dot(U.T)

theta, theta_deg = get_rotation_angle(R)
theta_deg

(3611, 2)
(3596, 2)
(3611, 3596)


-0.016995497154798034

In [338]:
import ot

coordinatesA = X
# coordinatesA = pd.read_csv('_overlap=100_align.csv')
coordinatesB = Y

# print(coordinatesA)
# print(coordinatesB)

coordinatesA = np.array(coordinatesA)
coordinatesB = np.array(coordinatesB)

disMatrix = ot.dist(coordinatesA, coordinatesB, metric='euclidean')
gt = pd.read_csv('./simData-20241219T155123Z-001/_overlap=100_mapping_matrix_ground_truth_spotIndex.csv', index_col=False)
gt = gt.iloc[:, 1:].reset_index(drop=True)
gt = gt.to_numpy()
s = (gt * disMatrix).sum()
# s
disPerSpot = s / coordinatesB.shape[0]
print(s, disPerSpot)

326974.82025365887 90.92736936976053


In [339]:
gt_r_deg = [-1, -10.29, -17.48, -26.47, -38.3]
st_r_deg = [-0.0169, -1.1857, -1.1713, 1.7306, -1.2868]

# # Convert to a NumPy array and create DataFrame
# r_array = np.array(st_r_deg)[:, np.newaxis] # localOT_deg is the rotation degree of localOT
# df = pd.DataFrame(r_array, columns=["localOT_r_deg"])
# df['pair'] = range(len(st_r_deg))

# Calculate absolute differences and add as a new column
differences = np.abs(np.array(st_r_deg) - np.array(gt_r_deg))
# df['dif_gt'] = differences

differences

array([ 0.9831,  9.1043, 16.3087, 28.2006, 37.0132])

In [340]:
# calculate SNA (spot number accuracy)
def cal_spotNoAcc(alignment, gt):
    # alignment = alignment.to_numpy()
    # gt = gt.to_numpy()
    rows = alignment.shape[0]
    cols = alignment.shape[1]
    adjusted_alignment = np.zeros((rows, cols))
    #print(np.min(alignment))
    for j in range(alignment.shape[0]):
        row = alignment[j]
        if np.max(row) -np.min(row) !=0:
            max_index = np.argmax(row)
            adjusted_alignment[j, max_index] = 1
    s = (adjusted_alignment * gt).sum()
    acc = s / gt.sum()
    return acc

sna = cal_spotNoAcc(alignment, gt)
sna

0.45383759733036705